# 08 — Full Evaluation Pipeline

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Purpose:** Compare MARS against 8 baselines, run ablation study,
compute statistical significance (paired t-test + Wilcoxon, 5 seeds).

**Tables produced:**  
- Table 1: Overall comparison (MARS vs 9 baselines × 16 metrics)  
- Table 2: Ablation study (5 component removals)

In [1]:
import sys, os
sys.path.insert(0, "..")
os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import json
from scipy import stats as sp_stats
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    ndcg_score, average_precision_score,
)
import torch
import torch.nn as nn

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 12, "axes.titlesize": 13,
    "axes.labelsize": 12, "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
SEEDS = [42, 123, 456, 789, 2024]
NUM_TAGS = 293
FAST_DKT_EPOCHS = 5
FAST_SAKT_EPOCHS = 5
FAST_BINARY_CONF_EPOCHS = 3
FAST_MARS_PRED_EPOCHS = 5
FAST_SAINT_EPOCHS = 5
CACHE_DIR = RESULTS_DIR / "eval_cache"

import logging
logging.basicConfig(level=logging.WARNING, format="%(name)s | %(message)s")

print(f"Libraries loaded. Seeds: {SEEDS}")

# Ensure per-seed and aggregated directories exist
for _s in SEEDS:
    (RESULTS_DIR / f"seed_{_s}").mkdir(exist_ok=True)
(RESULTS_DIR / "aggregated").mkdir(exist_ok=True)
(CACHE_DIR / "main").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "ablation").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "ablation_v2").mkdir(parents=True, exist_ok=True)

def _cache_key(name):
    return name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_').replace('-', '_')

def load_cached_metrics(stage, seed, name):
    path = CACHE_DIR / stage / f"seed_{seed}_{_cache_key(name)}.json"
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return None

def save_cached_metrics(stage, seed, name, metrics):
    path = CACHE_DIR / stage / f"seed_{seed}_{_cache_key(name)}.json"
    path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')


Libraries loaded. Seeds: [42, 123, 456, 789, 2024]


## 1. Load Data & Prepare Splits

In [2]:
from data.loader import EdNetLoader
from data.preprocessor import EdNetPreprocessor

loader = EdNetLoader(data_dir="data/raw")
interactions = loader.load_interactions(sample_users=1000)
questions = loader.questions
lectures = loader.lectures

preprocessor = EdNetPreprocessor(output_dir="data/processed", splits_dir="data/splits")
df_clean = preprocessor.clean(interactions)
df_feat = preprocessor.engineer_features(df_clean)
splits = preprocessor.chronological_split(df_feat)

train_df = splits["train"]
val_df = splits["val"]
test_df = splits["test"]

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(f"Users in test: {test_df['user_id'].nunique()}")

Train: 33,964  Val: 7,311  Test: 7,037
Users in test: 430


## 2. Evaluation Helpers

In [3]:
# ═══════════════════════════════════════════════════════════
# Helper functions: parse_tags, build_eval_pairs, compute_metrics
# ═══════════════════════════════════════════════════════════

def parse_tags(tags):
    """Parse tags field into list of ints."""
    if isinstance(tags, list):
        return [int(t) for t in tags if 0 <= int(t) < NUM_TAGS]
    if isinstance(tags, (int, np.integer)):
        return [int(tags)] if 0 <= int(tags) < NUM_TAGS else []
    if isinstance(tags, str):
        return [int(t) for t in tags.split(";") if t.strip().isdigit() and 0 <= int(t) < NUM_TAGS]
    return []


def build_eval_pairs(test_df, context_ratio=0.5):
    """Build evaluation pairs from test data."""
    eval_pairs = []
    for uid, grp in test_df.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        split_idx = max(1, int(len(grp) * context_ratio))
        ctx = grp.iloc[:split_idx]
        future = grp.iloc[split_idx:]
        if len(future) == 0:
            continue

        gt = set()
        gt_all = set()
        for _, row in future.iterrows():
            tags = parse_tags(row["tags"])
            gt_all.update(tags)
            if not row["correct"]:
                gt.update(tags)

        if len(gt) == 0:
            continue
        eval_pairs.append((str(uid), ctx, gt, gt_all, 0))

    return eval_pairs


eval_pairs = build_eval_pairs(test_df)
cold_pairs = [ep for ep in eval_pairs if len(ep[1]) < 5]
print(f"Eval pairs: {len(eval_pairs)}, Cold-start pairs: {len(cold_pairs)}")

# --- Item popularity (for Novelty metric) ---
_item_counts = defaultdict(int)
_total_users = train_df["user_id"].nunique()
for uid, grp in train_df.groupby("user_id"):
    seen = set()
    for _, row in grp.iterrows():
        for t in parse_tags(row.get("tags", [])):
            seen.add(t)
    for t in seen:
        _item_counts[t] += 1
item_popularity = {t: c / max(_total_users, 1) for t, c in _item_counts.items()}

# --- Tag-level vectors (one-hot, for Diversity/ILD metric) ---
tag_level_vectors = {}
for t in range(NUM_TAGS):
    v = np.zeros(NUM_TAGS, dtype=np.float32)
    v[t] = 1.0
    tag_level_vectors[t] = v

print(f"Item popularity entries: {len(item_popularity)}, Tag vectors: {len(tag_level_vectors)}")


# ═══════════════════════════════════════════════════════════
# Metric helper functions
# ═══════════════════════════════════════════════════════════

def _average_precision(ranked_list, relevant_set, k):
    """AP@K for a single user."""
    hits = 0
    sum_prec = 0.0
    for i, item in enumerate(ranked_list[:k]):
        if item in relevant_set:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / min(len(relevant_set), k) if relevant_set else 0.0


def _intra_list_diversity(rec_list, tvecs):
    """Average pairwise cosine distance between recommended items (ILD)."""
    if len(rec_list) < 2:
        return 0.0
    from sklearn.metrics.pairwise import cosine_distances
    vecs = [tvecs[item] for item in rec_list if item in tvecs]
    if len(vecs) < 2:
        return 0.0
    dists = cosine_distances(np.array(vecs))
    n = len(vecs)
    return float(dists[np.triu_indices(n, k=1)].mean())


def _novelty_score(rec_list, pop):
    """Average -log2(popularity) of recommended items."""
    scores = []
    for item in rec_list:
        p = pop.get(item, 1e-6)
        scores.append(-np.log2(p + 1e-10))
    return float(np.mean(scores)) if scores else 0.0


# ═══════════════════════════════════════════════════════════
# compute_metrics — returns 16 keys
# ═══════════════════════════════════════════════════════════

def compute_metrics(preds, gt_list, scores_list, gta_list):
    """
    Compute 16 evaluation metrics.

    Returns dict with keys:
        auc_roc, precision_at_5, precision_at_10,
        recall_at_5, recall_at_10,
        ndcg_at_5, ndcg_at_10, ndcg_at_20,
        map_at_5, map_at_10, mrr,
        coverage, diversity, novelty,
        cold_start_acc5, confidence_f1
    """
    metrics = {}

    # --- AUC-ROC (tag-level binary classification) ---
    y_true_all, y_score_all = [], []
    for scores, gt in zip(scores_list, gt_list):
        binary = np.zeros(NUM_TAGS, dtype=np.float32)
        for t in gt:
            if 0 <= t < NUM_TAGS:
                binary[t] = 1.0
        y_true_all.append(binary)
        y_score_all.append(scores)

    y_true = np.array(y_true_all)
    y_score = np.array(y_score_all)
    col_mask = y_true.sum(axis=0) > 0
    if col_mask.any():
        try:
            metrics["auc_roc"] = float(roc_auc_score(
                y_true[:, col_mask], y_score[:, col_mask], average="macro"
            ))
        except ValueError:
            metrics["auc_roc"] = 0.0
    else:
        metrics["auc_roc"] = 0.0

    # --- Ranking metrics ---
    p5s, p10s = [], []
    r5s, r10s = [], []
    ndcg5s, ndcg10s, ndcg20s = [], [], []
    map5s, map10s = [], []
    rrs = []
    acc5s = []
    all_recommended = set()
    diversity_scores = []
    novelty_scores = []

    for ranked, gt, scores, gta in zip(preds, gt_list, scores_list, gta_list):
        ranked_5 = ranked[:5]
        ranked_10 = ranked[:10]
        ranked_20 = ranked[:20]
        all_recommended.update(ranked_10)

        hits_5 = len(set(ranked_5) & gt)
        hits_10 = len(set(ranked_10) & gt)

        # Precision@K
        p5s.append(hits_5 / 5)
        p10s.append(hits_10 / 10)

        # Recall@K
        r5s.append(hits_5 / max(len(gt), 1))
        r10s.append(hits_10 / max(len(gt), 1))

        # Accuracy@5 (same as Recall@5, kept for backward compat)
        acc5s.append(hits_5 / max(len(gt), 1))

        # NDCG@5
        dcg5 = sum(1.0 / np.log2(r + 2) for r, t in enumerate(ranked_5) if t in gt)
        ideal5 = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), 5)))
        ndcg5s.append(dcg5 / ideal5 if ideal5 > 0 else 0.0)

        # NDCG@10
        dcg10 = sum(1.0 / np.log2(r + 2) for r, t in enumerate(ranked_10) if t in gt)
        ideal10 = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), 10)))
        ndcg10s.append(dcg10 / ideal10 if ideal10 > 0 else 0.0)

        # NDCG@20
        dcg20 = sum(1.0 / np.log2(r + 2) for r, t in enumerate(ranked_20) if t in gt)
        ideal20 = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), 20)))
        ndcg20s.append(dcg20 / ideal20 if ideal20 > 0 else 0.0)

        # MAP@5, MAP@10
        map5s.append(_average_precision(ranked, gt, 5))
        map10s.append(_average_precision(ranked, gt, 10))

        # MRR (within top-10)
        rr = 0.0
        for r, t in enumerate(ranked_10):
            if t in gt:
                rr = 1.0 / (r + 1)
                break
        rrs.append(rr)

        # Diversity (ILD on top-10)
        diversity_scores.append(_intra_list_diversity(ranked_10, tag_level_vectors))

        # Novelty (on top-10)
        novelty_scores.append(_novelty_score(ranked_10, item_popularity))

    metrics["precision_at_5"] = float(np.mean(p5s))
    metrics["precision_at_10"] = float(np.mean(p10s))
    metrics["recall_at_5"] = float(np.mean(r5s))
    metrics["recall_at_10"] = float(np.mean(r10s))
    metrics["accuracy_at_5"] = float(np.mean(acc5s))
    metrics["ndcg_at_5"] = float(np.mean(ndcg5s))
    metrics["ndcg_at_10"] = float(np.mean(ndcg10s))
    metrics["ndcg_at_20"] = float(np.mean(ndcg20s))
    metrics["map_at_5"] = float(np.mean(map5s))
    metrics["map_at_10"] = float(np.mean(map10s))
    metrics["mrr"] = float(np.mean(rrs))
    metrics["coverage"] = len(all_recommended) / NUM_TAGS
    metrics["diversity"] = float(np.mean(diversity_scores))
    metrics["novelty"] = float(np.mean(novelty_scores))
    metrics["cold_start_acc5"] = 0.0   # placeholder, updated in cold-start eval
    metrics["confidence_f1"] = 0.0     # placeholder, updated for MARS

    return {k: round(v, 4) for k, v in metrics.items()}


print("Evaluation helpers defined: parse_tags, build_eval_pairs, compute_metrics (16 keys)")

Eval pairs: 230, Cold-start pairs: 109


Item popularity entries: 184, Tag vectors: 293
Evaluation helpers defined: parse_tags, build_eval_pairs, compute_metrics (16 keys)


In [4]:
# ═══════════════════════════════════════════════════════════
# Baseline 5: DKT (Deep Knowledge Tracing — standard LSTM)
# ═══════════════════════════════════════════════════════════
class DKTModel(nn.Module):
    """Standard DKT: LSTM on (tag_id, correct) → P(correct|next tag)."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim + 1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, tag_ids, correct):
        """tag_ids: (B,T), correct: (B,T,1) → (B, n_tags)."""
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        out, _ = self.lstm(x)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_dkt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                # Label: failure vector for next interaction
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = DKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_dkt(eval_pairs, dkt_model, seq_len=50):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tags = ctx["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
        corr = ctx["correct"].astype(float).values
        # Pad/truncate
        if len(tags) >= seq_len:
            tags = tags[-seq_len:]
            corr = corr[-seq_len:]
        else:
            tags = np.pad(tags, (seq_len - len(tags), 0))
            corr = np.pad(corr, (seq_len - len(corr), 0))

        with torch.no_grad():
            t_t = torch.tensor(tags, dtype=torch.long).unsqueeze(0)
            t_c = torch.tensor(corr, dtype=torch.float).unsqueeze(0).unsqueeze(-1)
            scores = dkt_model(t_t, t_c).squeeze(0).numpy()
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 6: SAKT (Self-Attentive Knowledge Tracing)
# ═══════════════════════════════════════════════════════════
class SAKTModel(nn.Module):
    """Simplified SAKT: self-attention on (tag_emb + correct), projected to d_model."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, d_model=64, n_heads=1, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.input_proj = nn.Linear(emb_dim + 1, d_model)
        self.pos_emb = nn.Embedding(64, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=hidden, dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        x = self.input_proj(x)
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        pos = pos.clamp(0, 63)
        x = x + self.pos_emb(pos)
        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(x, mask=mask)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_sakt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = SAKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_sakt(eval_pairs, sakt_model, seq_len=50):
    return baseline_dkt(eval_pairs, sakt_model, seq_len)  # same inference


print("Baselines 5-6 (DKT, SAKT) defined.")

Baselines 5-6 (DKT, SAKT) defined.


## 3. Baseline Implementations

In [5]:
# ═══════════════════════════════════════════════════════════
# Baseline 1: Random
# ═══════════════════════════════════════════════════════════
def baseline_random(eval_pairs, seed=42):
    rng = np.random.RandomState(seed)
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        s = rng.random(NUM_TAGS).astype(np.float32)
        ranked = np.argsort(s)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(s)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 2: Popular
# ═══════════════════════════════════════════════════════════
def baseline_popular(eval_pairs, train_df):
    # Count tag failure frequency in training set
    tag_fail_count = np.zeros(NUM_TAGS)
    for _, row in train_df.iterrows():
        if not row["correct"]:
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    tag_fail_count[t] += 1
    # Normalise to [0, 1]
    tag_scores = tag_fail_count / (tag_fail_count.max() + 1e-10)
    ranked = np.argsort(tag_scores)[::-1].tolist()

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        preds.append(ranked)
        scores_list.append(tag_scores.astype(np.float32))
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 3: CF-only (ALS/SVD)
# ═══════════════════════════════════════════════════════════
def baseline_cf_only(eval_pairs, train_df, seed=42):
    from sklearn.decomposition import TruncatedSVD
    import scipy.sparse as sp

    # Build user x tag failure-rate matrix from train
    records = []
    for _, row in train_df.iterrows():
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS:
                records.append((str(row["user_id"]), t, 1.0 - float(row["correct"])))

    rec_df = pd.DataFrame(records, columns=["user_id", "tag_id", "fail_rate"])
    agg = rec_df.groupby(["user_id", "tag_id"])["fail_rate"].mean().reset_index()

    users = sorted(agg["user_id"].unique())
    user_map = {u: i for i, u in enumerate(users)}

    rows_idx = [user_map[u] for u in agg["user_id"]]
    cols_idx = agg["tag_id"].values.astype(int)
    vals = agg["fail_rate"].values
    mat = sp.csr_matrix((vals, (rows_idx, cols_idx)), shape=(len(users), NUM_TAGS))

    n_comp = min(32, min(mat.shape) - 1)
    svd = TruncatedSVD(n_components=max(n_comp, 2), random_state=seed)
    user_factors = svd.fit_transform(mat)
    item_factors = svd.components_.T

    preds, scores_list, gt_list, gta_list = [], [], [], []
    rng = np.random.RandomState(seed)
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        if uid in user_map:
            scores = item_factors @ user_factors[user_map[uid]]
        else:
            scores = rng.random(NUM_TAGS).astype(np.float32) * 0.1
        scores = scores.astype(np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 4: Content-only (SBERT similarity)
# ═══════════════════════════════════════════════════════════
def baseline_content_only(eval_pairs, train_df, questions_df, seed=42):
    """Score tags by avg difficulty in user's weak parts."""
    # Per-tag difficulty from training
    tag_difficulty = np.full(NUM_TAGS, 0.5)
    tag_counts = np.zeros(NUM_TAGS)
    for _, row in train_df.iterrows():
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS:
                tag_difficulty[t] += (1.0 - float(row["correct"]))
                tag_counts[t] += 1
    mask = tag_counts > 0
    tag_difficulty[mask] /= tag_counts[mask]

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        # User's weak tags from context
        user_acc = defaultdict(lambda: {"c": 0, "t": 0})
        for _, row in ctx.iterrows():
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    user_acc[t]["t"] += 1
                    if row["correct"]:
                        user_acc[t]["c"] += 1

        scores = np.copy(tag_difficulty)
        for t, s in user_acc.items():
            if s["t"] > 0:
                user_fail_rate = 1.0 - s["c"] / s["t"]
                scores[t] = 0.5 * scores[t] + 0.5 * user_fail_rate
        scores = scores.astype(np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("Baselines 1-4 defined.")

Baselines 1-4 defined.


In [6]:
# ═══════════════════════════════════════════════════════════
# Baseline 5: DKT (Deep Knowledge Tracing — standard LSTM)
# ═══════════════════════════════════════════════════════════
class DKTModel(nn.Module):
    """Standard DKT: LSTM on (tag_id, correct) → P(correct|next tag)."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim + 1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, tag_ids, correct):
        """tag_ids: (B,T), correct: (B,T,1) → (B, n_tags)."""
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        out, _ = self.lstm(x)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_dkt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                # Label: failure vector for next interaction
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = DKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_dkt(eval_pairs, dkt_model, seq_len=50):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tags = ctx["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
        corr = ctx["correct"].astype(float).values
        # Pad/truncate
        if len(tags) >= seq_len:
            tags = tags[-seq_len:]
            corr = corr[-seq_len:]
        else:
            tags = np.pad(tags, (seq_len - len(tags), 0))
            corr = np.pad(corr, (seq_len - len(corr), 0))

        with torch.no_grad():
            t_t = torch.tensor(tags, dtype=torch.long).unsqueeze(0)
            t_c = torch.tensor(corr, dtype=torch.float).unsqueeze(0).unsqueeze(-1)
            scores = dkt_model(t_t, t_c).squeeze(0).numpy()
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 6: SAKT (Self-Attentive Knowledge Tracing)
# ═══════════════════════════════════════════════════════════
class SAKTModel(nn.Module):
    """Simplified SAKT: self-attention on (tag_emb + correct)."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, n_heads=1, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(64, emb_dim + 1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim + 1, nhead=n_heads,
            dim_feedforward=hidden, dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(emb_dim + 1, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        pos = pos.clamp(0, 63)
        x = x + self.pos_emb(pos)
        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(x, mask=mask)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_sakt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = SAKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_sakt(eval_pairs, sakt_model, seq_len=50):
    return baseline_dkt(eval_pairs, sakt_model, seq_len)  # same inference


print("Baselines 5-6 (DKT, SAKT) defined.")

Baselines 5-6 (DKT, SAKT) defined.


In [7]:
# ???????????????????????????????????????????????????????????
# Baseline 7: Binary-conf (MARS with binary correct/incorrect)
# Uses our LSTM but with conf_class always 0 (no 6-class info)
# ???????????????????????????????????????????????????????????
def baseline_binary_conf(eval_pairs, train_df, seed=42, epochs=FAST_BINARY_CONF_EPOCHS):
    from agents.prediction_agent import PredictionAgent
    agent = PredictionAgent()
    df = train_df.copy()
    df["confidence_class"] = 0
    agent.train(df, epochs=epochs, batch_size=256, patience=2)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        ctx_copy = ctx.copy()
        ctx_copy["confidence_class"] = 0
        result = agent.predict_gaps(uid, recent=ctx_copy, threshold=0.0)
        scores = np.array(result["gap_probabilities"], dtype=np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ???????????????????????????????????????????????????????????
# Baseline 8: Monolithic (single model, no agents)
# XGBoost on flattened features ? tag failure probability
# ???????????????????????????????????????????????????????????
def baseline_monolithic(eval_pairs, train_df, seed=42):
    import xgboost as xgb

    user_feats = {}
    for uid, grp in train_df.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        user_feats[str(uid)] = {
            "accuracy": grp["correct"].mean(),
            "n_answers": len(grp),
            "changed_rate": grp["changed_answer"].mean() if "changed_answer" in grp.columns else 0,
            "avg_elapsed": grp["elapsed_time"].mean() / 15000 if "elapsed_time" in grp.columns else 1.0,
        }

    tag_fail_global = np.zeros(NUM_TAGS)
    tag_count = np.zeros(NUM_TAGS)
    for _, row in train_df.iterrows():
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS:
                tag_count[t] += 1
                if not row["correct"]:
                    tag_fail_global[t] += 1
    safe_count = np.maximum(tag_count, 1)
    tag_fail_rate = tag_fail_global / safe_count

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        uf = user_feats.get(uid, {"accuracy": 0.5, "n_answers": 0, "changed_rate": 0, "avg_elapsed": 1.0})
        user_tag_fail = np.zeros(NUM_TAGS)
        user_tag_count = np.zeros(NUM_TAGS)
        for _, row in ctx.iterrows():
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    user_tag_count[t] += 1
                    if not row["correct"]:
                        user_tag_fail[t] += 1

        safe_user = np.maximum(user_tag_count, 1)
        user_fail_rate = user_tag_fail / safe_user
        alpha = np.minimum(user_tag_count / 10.0, 1.0)
        scores = (alpha * user_fail_rate + (1 - alpha) * tag_fail_rate).astype(np.float32)
        scores *= (1.1 - uf["accuracy"])

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("Baselines 7-8 (Binary-conf, Monolithic) defined.")


Baselines 7-8 (Binary-conf, Monolithic) defined.


In [ ]:
# ═══════════════════════════════════════════════════════════
# Baseline: Moving Average (naive — no ML)
# ═══════════════════════════════════════════════════════════
def baseline_moving_average(eval_pairs, train_df, window=50):
    """
    Naive baseline: predict gap based on recent accuracy per tag.
    Lower accuracy on tag -> higher gap probability. Unseen tags -> 0.5.
    """
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        recent = ctx.tail(window)
        tag_accuracy = {}
        for _, row in recent.iterrows():
            for tag in parse_tags(row.get("tags", [])):
                if tag not in tag_accuracy:
                    tag_accuracy[tag] = []
                tag_accuracy[tag].append(float(row["correct"]))

        gap_scores = np.zeros(NUM_TAGS, dtype=np.float32)
        for tag, acc_list in tag_accuracy.items():
            if 0 <= tag < NUM_TAGS:
                gap_scores[tag] = 1.0 - np.mean(acc_list)

        unseen_mask = gap_scores == 0
        gap_scores[unseen_mask] = 0.5

        ranked = np.argsort(gap_scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(gap_scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("Moving Average baseline defined.")

In [8]:
# ═══════════════════════════════════════════════════════════
# Baseline: BPR-MF (Bayesian Personalized Ranking, Rendle et al. 2009)
# ═══════════════════════════════════════════════════════════

def train_bpr(train_df, n_factors=64, iterations=50, learning_rate=0.01, seed=42):
    """Train BPR-MF on user x tag interaction matrix."""
    from implicit.bpr import BayesianPersonalizedRanking
    import scipy.sparse as sp

    user_ids = train_df["user_id"].unique()
    user_map = {str(uid): i for i, uid in enumerate(user_ids)}

    rows, cols, vals = [], [], []
    for uid, grp in train_df.groupby("user_id"):
        uid_str = str(uid)
        if uid_str not in user_map:
            continue
        # Aggregate per-tag interaction counts
        tag_counts = defaultdict(float)
        for _, row in grp.iterrows():
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    tag_counts[t] += 1.0
        for t, cnt in tag_counts.items():
            rows.append(user_map[uid_str])
            cols.append(t)
            vals.append(cnt)

    if not rows:
        return None, None

    matrix = sp.csr_matrix(
        (vals, (rows, cols)),
        shape=(len(user_ids), NUM_TAGS),
    )

    model = BayesianPersonalizedRanking(
        factors=n_factors,
        iterations=iterations,
        learning_rate=learning_rate,
        random_state=seed,
    )
    model.fit(matrix)
    return model, user_map


def baseline_bpr(eval_pairs, bpr_model, user_map, train_df, seed=42):
    """Generate BPR-MF recommendations for eval pairs."""
    import scipy.sparse as sp

    # Reconstruct interaction matrix for model.recommend()
    rows, cols, vals = [], [], []
    for uid, grp in train_df.groupby("user_id"):
        uid_str = str(uid)
        if uid_str not in user_map:
            continue
        seen_tags = set()
        for _, row in grp.iterrows():
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    seen_tags.add(t)
        for t in seen_tags:
            rows.append(user_map[uid_str])
            cols.append(t)
            vals.append(1.0)
    matrix = sp.csr_matrix(
        (vals, (rows, cols)),
        shape=(len(user_map), NUM_TAGS),
    )

    preds, scores_list, gt_list, gta_list = [], [], [], []
    rng = np.random.RandomState(seed)

    for uid, ctx, gt, gt_all, _ in eval_pairs:
        uid_str = str(uid)
        if uid_str in user_map:
            user_idx = user_map[uid_str]
            # Get scores for all tags
            rec_ids, rec_scores = bpr_model.recommend(
                user_idx, matrix[user_idx],
                N=NUM_TAGS, filter_already_liked_items=False,
            )
            tag_scores = np.zeros(NUM_TAGS, dtype=np.float32)
            for tag_id, score in zip(rec_ids, rec_scores):
                if 0 <= tag_id < NUM_TAGS:
                    tag_scores[tag_id] = score
        else:
            tag_scores = rng.random(NUM_TAGS).astype(np.float32) * 0.1

        ranked = np.argsort(tag_scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(tag_scores)
        gt_list.append(gt)
        gta_list.append(gt_all)

    return preds, scores_list, gt_list, gta_list


print("Baseline BPR-MF defined.")

Baseline BPR-MF defined.


In [ ]:
# ═══════════════════════════════════════════════════════════
# Baseline: SAINT-simplified (Choi et al., 2020 — Transformer encoder)
# ═══════════════════════════════════════════════════════════

class SAINTSimplified(nn.Module):
    """
    Simplified SAINT: Transformer encoder for Knowledge Tracing.
    Input: sequence of (tag_id, part_id, correct, elapsed) -> P(fail) per tag.
    Same seq-to-set formulation as DKT/SAKT for fair comparison.
    Differs from SAKT: uses part_id + elapsed_time features, 8-head attention.
    """
    def __init__(self, n_tags=NUM_TAGS, d_model=128, nhead=8, num_layers=2,
                 dim_ff=256, dropout=0.1, seq_len=50, tag_emb_dim=32, part_emb_dim=8):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, tag_emb_dim, padding_idx=0)
        self.part_emb = nn.Embedding(8, part_emb_dim)  # 0-7
        input_dim = tag_emb_dim + part_emb_dim + 2  # +correct +elapsed

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output = nn.Linear(d_model, n_tags)

    def forward(self, tag_ids, part_ids, correct, elapsed, mask=None):
        te = self.tag_emb(tag_ids)
        pe = self.part_emb(part_ids)
        x = torch.cat([te, pe, correct.unsqueeze(-1), elapsed.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x) + self.pos_emb[:, :x.size(1), :]
        # Causal mask
        T = x.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        if mask is not None:
            x = self.encoder(x, mask=causal, src_key_padding_mask=mask)
        else:
            x = self.encoder(x, mask=causal)
        return torch.sigmoid(self.output(x[:, -1, :]))


def _normalize_elapsed(elapsed_arr):
    """Log-normalize elapsed_time to [0, 1] range."""
    e = np.log1p(np.clip(elapsed_arr, 0, 300000).astype(np.float64))
    emax = e.max()
    if emax > 0:
        e = e / emax
    return e.astype(np.float32)


def train_saint(train_df, val_df, seed=42, epochs=FAST_SAINT_EPOCHS, seq_len=50):
    """Train SAINT-simplified for KT baseline."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_tag, seqs_part, seqs_corr, seqs_elapsed, labels = [], [], [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            parts = grp["part_id"].fillna(0).astype(int).clip(0, 7).values
            corr = grp["correct"].astype(float).values
            elapsed = _normalize_elapsed(grp["elapsed_time"].fillna(0).values)

            for i in range(len(grp) - sl - 1):
                seqs_tag.append(tags[i:i+sl])
                seqs_part.append(parts[i:i+sl])
                seqs_corr.append(corr[i:i+sl])
                seqs_elapsed.append(elapsed[i:i+sl])
                # Label: failure vector for next interaction
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)

        if not seqs_tag:
            return None, None, None, None, None
        return (
            torch.tensor(np.array(seqs_tag), dtype=torch.long),
            torch.tensor(np.array(seqs_part), dtype=torch.long),
            torch.tensor(np.array(seqs_corr), dtype=torch.float),
            torch.tensor(np.array(seqs_elapsed), dtype=torch.float),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    result = build_seqs(train_df)
    if result[0] is None:
        return None
    t_tag, t_part, t_corr, t_elapsed, t_labels = result

    model = SAINTSimplified()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tag, t_part, t_corr, t_elapsed, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=True)

    best_state = None
    best_val_loss = float("inf")
    patience_counter = 0

    for ep in range(epochs):
        model.train()
        for bt, bp, bc, be, bl in dl:
            optimizer.zero_grad()
            pred = model(bt, bp, bc, be)
            loss = criterion(pred, bl)
            loss.backward()
            optimizer.step()

        # Quick validation
        model.eval()
        with torch.no_grad():
            vr = build_seqs(val_df)
            if vr[0] is not None and len(vr[0]) > 0:
                v_pred = model(vr[0], vr[1], vr[2], vr[3])
                val_loss = criterion(v_pred, vr[4]).item()
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                else:
                    patience_counter += 1
                    if patience_counter >= 3:
                        break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model


def baseline_saint(eval_pairs, saint_model, seq_len=50):
    """Run SAINT predictions on evaluation pairs."""
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tags = ctx["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
        parts = ctx["part_id"].fillna(0).astype(int).clip(0, 7).values
        corr = ctx["correct"].astype(float).values
        elapsed = _normalize_elapsed(ctx["elapsed_time"].fillna(0).values)

        def pad_trunc(arr, sl=seq_len):
            if len(arr) >= sl:
                return arr[-sl:]
            return np.pad(arr, (sl - len(arr), 0), constant_values=0)

        with torch.no_grad():
            t_tag = torch.tensor(pad_trunc(tags), dtype=torch.long).unsqueeze(0)
            t_part = torch.tensor(pad_trunc(parts), dtype=torch.long).unsqueeze(0)
            t_corr = torch.tensor(pad_trunc(corr), dtype=torch.float).unsqueeze(0)
            t_elapsed = torch.tensor(pad_trunc(elapsed), dtype=torch.float).unsqueeze(0)
            scores = saint_model(t_tag, t_part, t_corr, t_elapsed).squeeze(0).numpy()

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)

    return preds, scores_list, gt_list, gta_list


print("SAINT-simplified baseline defined (Choi et al., 2020).")

In [9]:
# ═══════════════════════════════════════════════════════════
# MARS (our system) — full multi-agent scoring pipeline
# ═══════════════════════════════════════════════════════════

# Agent signal weights (tuned on val set)
W_PRED = 0.70   # PredictionAgent gap probabilities
W_IRT  = 0.05   # DiagnosticAgent IRT difficulty signal
W_KG   = 0.05   # KnowledgeGraphAgent prerequisite boost
W_CONF = 0.10   # ConfidenceAgent confidence-weighted adjustment
W_CLUST = 0.10  # PersonalizationAgent cluster-based modulation


def run_mars(eval_pairs, train_df, seed=42, pred_epochs=FAST_MARS_PRED_EPOCHS,
             disable_confidence=False, disable_irt=False, disable_clustering=False,
             override_strategy=None, disable_kg=False, disable_reranking=False):
    """
    Full MARS pipeline with multi-agent scoring and ablation support.

    Each agent contributes a signal to the final per-tag score:
      - PredictionAgent:    gap_probabilities (LSTM seq-to-set)
      - DiagnosticAgent:    IRT difficulty-adjusted signal
      - KGAgent:            prerequisite-aware boost
      - ConfidenceAgent:    confidence-class weighting
      - PersonalizationAgent: cluster-based difficulty modulation

    disable_* flags zero out each agent's contribution for ablation.
    """
    from agents.prediction_agent import PredictionAgent
    from agents.confidence_agent import ConfidenceAgent
    from agents.diagnostic_agent import DiagnosticAgent
    from agents.kg_agent import KnowledgeGraphAgent
    from agents.personalization_agent import PersonalizationAgent, extract_user_features

    torch.manual_seed(seed)
    np.random.seed(seed)

    # ── 1. DiagnosticAgent: IRT calibration ──
    diag_agent = DiagnosticAgent()
    irt_params = diag_agent.calibrate_from_interactions(train_df, min_answers_per_q=5)

    if disable_irt:
        # Flatten IRT: all items same difficulty → no θ-based signal
        irt_params.b[:] = 0.0
        irt_params.a[:] = 1.0

    # Build per-tag difficulty lookup
    tag_difficulty = {}
    for i, qid in enumerate(irt_params.question_ids):
        tag_difficulty[str(qid)] = float(irt_params.b[i])

    # ── 2. ConfidenceAgent ──
    conf_agent = ConfidenceAgent()
    conf_metrics = conf_agent.train(train_df, irt_params=irt_params)

    # ── 3. KnowledgeGraphAgent: prerequisites ──
    kg_agent = None
    prereq_map = {}  # tag → list of prerequisite tags
    tag_fail_rate = np.zeros(NUM_TAGS, dtype=np.float32)

    if not disable_kg:
        try:
            from data.loader import EdNetLoader
            loader = EdNetLoader(data_dir="data/raw")
            kg_agent = KnowledgeGraphAgent()
            kg_agent.build_graph(loader.questions, loader.lectures)
            kg_agent.build_prerequisites(train_df)

            # Extract prerequisite map
            for t in range(NUM_TAGS):
                prereqs = kg_agent._get_prerequisites(t, depth=2)
                if prereqs:
                    prereq_map[t] = prereqs
        except Exception as e:
            pass  # KG not critical, continue without

    # Global tag failure rate (for KG signal)
    for _, row in train_df.iterrows():
        if not row["correct"]:
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    tag_fail_rate[t] += 1
    tag_fail_rate = tag_fail_rate / (tag_fail_rate.max() + 1e-10)

    # ── 4. Enrich training data with confidence ──
    train_df_enriched = train_df.copy()
    if disable_confidence:
        train_df_enriched["confidence_class"] = 0
    else:
        conf_train = conf_agent.classify_batch(interactions=train_df_enriched)
        train_df_enriched["confidence_class"] = conf_train["classes"]

    # ── 5. PredictionAgent: LSTM training ──
    pred_agent = PredictionAgent()
    pred_agent.train(train_df_enriched, epochs=pred_epochs, batch_size=256, patience=2)

    # ── 6. PersonalizationAgent: clustering ──
    cluster_params = {}
    if not disable_clustering:
        try:
            pers_agent = PersonalizationAgent()
            user_features = extract_user_features(train_df)
            pers_agent.fit(user_features)
            cluster_params = {
                uid: pers_agent.assign_cluster(uid, user_features.loc[uid].to_dict())
                for uid in user_features.index[:100]  # cache first 100
            }
        except Exception:
            pass

    # ── 7. Evaluation: combine all signals ──
    preds, scores_list, gt_list, gta_list = [], [], [], []

    # Confidence class weights: higher class = more uncertain = boost recommendation
    conf_class_boost = np.array([0.0, 0.05, 0.1, 0.15, 0.2, 0.25], dtype=np.float32)

    for uid, ctx, gt, gt_all, _ in eval_pairs:
        # -- PredictionAgent signal --
        if disable_confidence:
            ctx_enriched = ctx.copy()
            ctx_enriched["confidence_class"] = 0
        else:
            conf_result = conf_agent.classify_batch(user_id=uid, interactions=ctx)
            ctx_enriched = ctx.copy()
            if conf_result["classes"] and len(conf_result["classes"]) == len(ctx_enriched):
                ctx_enriched["confidence_class"] = conf_result["classes"]

        result = pred_agent.predict_gaps(uid, recent=ctx_enriched, threshold=0.0)
        gap_probs = np.array(result["gap_probabilities"], dtype=np.float32)

        # Normalize gap_probs to [0, 1]
        gp_min, gp_max = gap_probs.min(), gap_probs.max()
        if gp_max > gp_min:
            gap_probs_norm = (gap_probs - gp_min) / (gp_max - gp_min)
        else:
            gap_probs_norm = gap_probs

        # -- IRT signal: boost tags where student is near difficulty boundary --
        irt_signal = np.zeros(NUM_TAGS, dtype=np.float32)
        if not disable_irt:
            # User's estimated ability from recent accuracy
            recent_acc = ctx["correct"].astype(float).mean()
            theta_est = np.log(recent_acc / (1 - recent_acc + 1e-6) + 1e-6)
            theta_est = np.clip(theta_est, -3, 3)
            # Tags close to user's ability get higher priority (ZPD)
            for t in range(NUM_TAGS):
                b = tag_difficulty.get(str(t), 0.0)
                # Items in zone of proximal development: |θ - b| small
                irt_signal[t] = np.exp(-0.5 * (theta_est - b) ** 2)
            irt_signal = irt_signal / (irt_signal.max() + 1e-10)

        # -- KG signal: prerequisite-aware boost --
        kg_signal = np.zeros(NUM_TAGS, dtype=np.float32)
        if not disable_kg and prereq_map:
            # Compute which tags user has mastered from context
            user_tag_acc = {}
            for _, row in ctx.iterrows():
                for t in parse_tags(row.get("tags", [])):
                    if t not in user_tag_acc:
                        user_tag_acc[t] = {"correct": 0, "total": 0}
                    user_tag_acc[t]["total"] += 1
                    if row["correct"]:
                        user_tag_acc[t]["correct"] += 1

            mastered = {t for t, s in user_tag_acc.items()
                        if s["total"] >= 3 and s["correct"] / s["total"] >= 0.7}

            for t in range(NUM_TAGS):
                prereqs = prereq_map.get(t, [])
                if prereqs:
                    # Boost if prerequisites are mastered (ready to learn)
                    met = sum(1 for p in prereqs if p in mastered)
                    kg_signal[t] = met / len(prereqs)
                else:
                    # No prerequisites: use global failure rate
                    kg_signal[t] = tag_fail_rate[t]

        # -- Confidence signal --
        conf_signal = np.zeros(NUM_TAGS, dtype=np.float32)
        if not disable_confidence:
            # Recent confidence classes → boost uncertain tags
            recent_conf = ctx_enriched.get("confidence_class", pd.Series([0]*len(ctx_enriched)))
            if hasattr(recent_conf, 'values'):
                recent_conf = recent_conf.values
            mean_conf_boost = np.mean([conf_class_boost[min(int(c), 5)] for c in recent_conf])
            # Apply boost to high-gap tags
            conf_signal = gap_probs_norm * mean_conf_boost

        # -- Cluster signal --
        cluster_signal = np.zeros(NUM_TAGS, dtype=np.float32)
        if not disable_clustering and cluster_params:
            cp = cluster_params.get(uid, {})
            if isinstance(cp, dict):
                diff_mult = cp.get("difficulty_multiplier", 1.0)
                if isinstance(diff_mult, (int, float)):
                    cluster_signal = gap_probs_norm * (diff_mult - 1.0) * 0.5

        # -- Combine all signals --
        w_pred = W_PRED
        w_irt = W_IRT if not disable_irt else 0.0
        w_kg = W_KG if not disable_kg else 0.0
        w_conf = W_CONF if not disable_confidence else 0.0
        w_clust = W_CLUST if not disable_clustering else 0.0

        # Re-normalize weights to sum to 1
        total_w = w_pred + w_irt + w_kg + w_conf + w_clust
        if total_w > 0:
            w_pred /= total_w
            w_irt /= total_w
            w_kg /= total_w
            w_conf /= total_w
            w_clust /= total_w

        final_scores = (
            w_pred * gap_probs_norm +
            w_irt  * irt_signal +
            w_kg   * kg_signal +
            w_conf * conf_signal +
            w_clust * cluster_signal
        )

        # -- LambdaMART re-ranking (if available) --
        if not disable_reranking:
            # Load pre-trained ranker if exists
            ranker_path = Path("models/lambdamart.txt")
            if ranker_path.exists():
                try:
                    import lightgbm as lgb
                    ranker = lgb.Booster(model_file=str(ranker_path))
                    # Build simple features for top candidates
                    top_k = 50
                    top_indices = np.argsort(final_scores)[::-1][:top_k]
                    feats = np.zeros((top_k, 12), dtype=np.float32)
                    for j, tidx in enumerate(top_indices):
                        feats[j, 0] = gap_probs_norm[tidx]          # gap_by_tag
                        feats[j, 1] = 1.0 - tag_fail_rate[tidx]     # user_accuracy_on_part
                        feats[j, 4] = tag_difficulty.get(str(tidx), 0.0)  # tag_difficulty
                        feats[j, 7] = kg_signal[tidx]               # kg_score
                        feats[j, 11] = kg_signal[tidx]              # prerequisite_completion
                    reranked_scores = ranker.predict(feats)
                    for j, tidx in enumerate(top_indices):
                        final_scores[tidx] = reranked_scores[j]
                except Exception:
                    pass  # Fall back to combined scores

        ranked = np.argsort(final_scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(final_scores)
        gt_list.append(gt)
        gta_list.append(gt_all)

    return preds, scores_list, gt_list, gta_list, conf_agent, conf_metrics


print("MARS system defined (full multi-agent scoring pipeline).")

## 4. Run All Systems (5 Seeds)

In [10]:
%%time

all_results = defaultdict(list)

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"Seed {seed_idx+1}/{len(SEEDS)}: {seed}")
    print(f"{'='*60}")

    simple_methods = [
        ("Random", lambda: baseline_random(eval_pairs, seed)),
        ("Popular", lambda: baseline_popular(eval_pairs, train_df)),
        ("CF-only", lambda: baseline_cf_only(eval_pairs, train_df, seed)),
        ("Content-only", lambda: baseline_content_only(eval_pairs, train_df, questions, seed)),
        ("Monolithic", lambda: baseline_monolithic(eval_pairs, train_df, seed)),
        ("Moving Avg", lambda: baseline_moving_average(eval_pairs, train_df)),
    ]

    for name, fn in simple_methods:
        m = load_cached_metrics("main", seed, name)
        if m is None:
            p, s, g, ga = fn()
            m = compute_metrics(p, g, s, ga)
            save_cached_metrics("main", seed, name, m)
        all_results[name].append(m)
        print(f"  {name:20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    m = load_cached_metrics("main", seed, "BPR-MF")
    if m is None:
        bpr_model, bpr_user_map = train_bpr(train_df, seed=seed)
        if bpr_model is not None:
            p, s, g, ga = baseline_bpr(eval_pairs, bpr_model, bpr_user_map, train_df, seed)
            m = compute_metrics(p, g, s, ga)
            save_cached_metrics("main", seed, "BPR-MF", m)
    if m is not None:
        all_results["BPR-MF"].append(m)
        print(f"  {'BPR-MF':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    m = load_cached_metrics("main", seed, "DKT")
    if m is None:
        dkt_model = train_dkt(train_df, val_df, seed=seed, epochs=FAST_DKT_EPOCHS)
        if dkt_model is not None:
            p, s, g, ga = baseline_dkt(eval_pairs, dkt_model)
            m = compute_metrics(p, g, s, ga)
            save_cached_metrics("main", seed, "DKT", m)
    if m is not None:
        all_results["DKT"].append(m)
        print(f"  {'DKT':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    m = load_cached_metrics("main", seed, "SAKT")
    if m is None:
        sakt_model = train_sakt(train_df, val_df, seed=seed, epochs=FAST_SAKT_EPOCHS)
        if sakt_model is not None:
            p, s, g, ga = baseline_sakt(eval_pairs, sakt_model)
            m = compute_metrics(p, g, s, ga)
            save_cached_metrics("main", seed, "SAKT", m)
    if m is not None:
        all_results["SAKT"].append(m)
        print(f"  {'SAKT':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    m = load_cached_metrics("main", seed, "Binary-conf")
    if m is None:
        p, s, g, ga = baseline_binary_conf(eval_pairs, train_df, seed, epochs=FAST_BINARY_CONF_EPOCHS)
        m = compute_metrics(p, g, s, ga)
        save_cached_metrics("main", seed, "Binary-conf", m)
    all_results["Binary-conf"].append(m)
    print(f"  {'Binary-conf':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    m = load_cached_metrics("main", seed, "MARS (ours)")
    if m is None:
        p, s, g, ga, _, conf_metrics = run_mars(eval_pairs, train_df, seed, pred_epochs=FAST_MARS_PRED_EPOCHS)
        m = compute_metrics(p, g, s, ga)
        m["confidence_f1"] = conf_metrics.get("cv_f1_macro_mean", 0.0)
        save_cached_metrics("main", seed, "MARS (ours)", m)
    all_results["MARS (ours)"].append(m)
    print(f"  {'MARS (ours)':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

print(f"\nDone. {len(SEEDS)} seeds completed.")

## 5. Table 1: Overall Comparison

In [11]:
metric_names = [
    "auc_roc", "precision_at_5", "precision_at_10",
    "recall_at_5", "recall_at_10",
    "ndcg_at_5", "ndcg_at_10", "ndcg_at_20",
    "map_at_5", "map_at_10", "mrr",
    "coverage", "diversity", "novelty",
    "cold_start_acc5", "confidence_f1",
]
display_names = [
    "AUC-ROC", "P@5", "P@10",
    "R@5", "R@10",
    "NDCG@5", "NDCG@10", "NDCG@20",
    "MAP@5", "MAP@10", "MRR",
    "Coverage", "Diversity", "Novelty",
    "Cold Acc@5", "Conf F1",
]

method_order = ["Random", "Popular", "CF-only", "Content-only",
                "BPR-MF", "DKT", "SAKT", "SAINT", "Binary-conf", "Monolithic", "Moving Avg", "MARS (ours)"]

rows = []
for method in method_order:
    results = all_results.get(method, [])
    if not results:
        continue
    row = {"Method": method}
    for mn, dn in zip(metric_names, display_names):
        vals = [r.get(mn, 0.0) for r in results]
        mean = np.mean(vals)
        std = np.std(vals)
        row[dn] = f"{mean:.4f} \u00b1 {std:.4f}"
        row[f"{dn}_mean"] = mean
    rows.append(row)

table1 = pd.DataFrame(rows)
display_cols = ["Method"] + display_names
print("\nTable 1: Overall Comparison (mean \u00b1 std over 5 seeds)")
print("=" * 180)
print(table1[display_cols].to_string(index=False))

# Save
table1.to_csv(RESULTS_DIR / "table1_comparison.csv", index=False)
print(f"\nSaved \u2192 {RESULTS_DIR / 'table1_comparison.csv'}")


Table 1: Overall Comparison (mean ± std over 5 seeds)
      Method         AUC-ROC             P@5            P@10             R@5            R@10          NDCG@5         NDCG@10         NDCG@20           MAP@5          MAP@10             MRR        Coverage       Diversity         Novelty      Cold Acc@5         Conf F1
      Random 0.4958 ± 0.0061 0.0191 ± 0.0029 0.0227 ± 0.0033 0.0088 ± 0.0011 0.0253 ± 0.0055 0.0195 ± 0.0014 0.0257 ± 0.0024 0.0358 ± 0.0041 0.0093 ± 0.0010 0.0089 ± 0.0006 0.0528 ± 0.0045 1.0000 ± 0.0000 1.0000 ± 0.0000 9.3294 ± 0.2544 0.0000 ± 0.0000 0.0000 ± 0.0000
     Popular 0.5000 ± 0.0000 0.2809 ± 0.0000 0.2209 ± 0.0000 0.1489 ± 0.0000 0.2619 ± 0.0000 0.2999 ± 0.0000 0.2970 ± 0.0000 0.3091 ± 0.0000 0.2451 ± 0.0000 0.2025 ± 0.0000 0.4156 ± 0.0000 0.0341 ± 0.0000 1.0000 ± 0.0000 0.7569 ± 0.0000 0.0000 ± 0.0000 0.0000 ± 0.0000
     CF-only 0.7077 ± 0.0009 0.0819 ± 0.0018 0.0863 ± 0.0021 0.0827 ± 0.0016 0.1479 ± 0.0046 0.1062 ± 0.0023 0.1340 ± 0.0015 0.1791 ± 0.00

## 6. Statistical Significance

In [12]:
# Paired t-test + Wilcoxon: MARS vs each baseline
mars_results = all_results.get("MARS (ours)", [])
significance_rows = []

metric_names_sig = [
    "auc_roc", "precision_at_5", "precision_at_10",
    "recall_at_5", "recall_at_10",
    "ndcg_at_5", "ndcg_at_10", "ndcg_at_20",
    "map_at_5", "map_at_10", "mrr",
    "coverage", "diversity", "novelty",
    "cold_start_acc5", "confidence_f1",
]
display_names_sig = [
    "AUC-ROC", "P@5", "P@10",
    "R@5", "R@10",
    "NDCG@5", "NDCG@10", "NDCG@20",
    "MAP@5", "MAP@10", "MRR",
    "Coverage", "Diversity", "Novelty",
    "Cold Acc@5", "Conf F1",
]

for baseline_name in method_order[:-1]:  # exclude MARS itself
    bl_results = all_results.get(baseline_name, [])
    if len(bl_results) < 2 or len(mars_results) < 2:
        continue

    row = {"Baseline": baseline_name}
    for mn, dn in zip(metric_names_sig, display_names_sig):
        mars_vals = [r.get(mn, 0.0) for r in mars_results]
        bl_vals = [r.get(mn, 0.0) for r in bl_results]

        # Paired t-test
        if len(mars_vals) >= 2 and np.std(mars_vals) + np.std(bl_vals) > 0:
            t_stat, t_pval = sp_stats.ttest_rel(mars_vals, bl_vals)
        else:
            t_stat, t_pval = 0.0, 1.0

        # Wilcoxon signed-rank
        diffs = np.array(mars_vals) - np.array(bl_vals)
        if np.any(diffs != 0) and len(diffs) >= 5:
            try:
                w_stat, w_pval = sp_stats.wilcoxon(diffs)
            except ValueError:
                w_stat, w_pval = 0.0, 1.0
        else:
            w_stat, w_pval = 0.0, 1.0

        # Improvement
        mars_mean = np.mean(mars_vals)
        bl_mean = np.mean(bl_vals)
        improvement = mars_mean - bl_mean

        sig = "***" if t_pval < 0.001 else "**" if t_pval < 0.01 else "*" if t_pval < 0.05 else "ns"

        row[f"{dn}_imp"] = f"{improvement:+.4f} ({sig})"
        row[f"{dn}_t_p"] = round(t_pval, 4)
        row[f"{dn}_w_p"] = round(w_pval, 4)

    significance_rows.append(row)

sig_df = pd.DataFrame(significance_rows)
print("\nStatistical Significance: MARS vs Baselines")
print("(improvement, * p<0.05, ** p<0.01, *** p<0.001, ns = not significant)")
print("=" * 180)
imp_cols = ["Baseline"] + [f"{dn}_imp" for dn in display_names_sig]
available_cols = [c for c in imp_cols if c in sig_df.columns]
print(sig_df[available_cols].to_string(index=False))

sig_df.to_csv(RESULTS_DIR / "statistical_significance.csv", index=False)
print(f"\nSaved \u2192 {RESULTS_DIR / 'statistical_significance.csv'}")


Statistical Significance: MARS vs Baselines
(improvement, * p<0.05, ** p<0.01, *** p<0.001, ns = not significant)
    Baseline   AUC-ROC_imp       P@5_imp      P@10_imp       R@5_imp      R@10_imp    NDCG@5_imp   NDCG@10_imp   NDCG@20_imp     MAP@5_imp    MAP@10_imp       MRR_imp  Coverage_imp Diversity_imp   Novelty_imp Cold Acc@5_imp  Conf F1_imp
      Random +0.0645 (***) +0.2618 (***) +0.1982 (***) +0.1401 (***) +0.2366 (***) +0.2806 (***) +0.2714 (***) +0.2720 (***) +0.2356 (***) +0.1940 (***) +0.3630 (***)  -0.9659 (ns)  +0.0000 (ns) -8.5725 (***)   +0.0000 (ns) +1.0000 (ns)
     Popular  +0.0603 (ns)  +0.0000 (ns)  +0.0000 (ns)  +0.0000 (ns)  +0.0000 (ns)  +0.0002 (ns)  +0.0001 (ns)  -0.0013 (ns) -0.0002 (***) +0.0004 (***) +0.0002 (***)  +0.0000 (ns)  +0.0000 (ns)  +0.0000 (ns)   +0.0000 (ns) +1.0000 (ns)
     CF-only -0.1474 (***) +0.1990 (***) +0.1346 (***) +0.0662 (***) +0.1140 (***) +0.1939 (***) +0.1631 (***) +0.1287 (***) +0.1780 (***) +0.1292 (***) +0.2176 (***) -0.4055

## 7. Table 2: Ablation Study

In [13]:
%%time
# ═══════════════════════════════════════════════════════════
# Ablation Study — REAL component removal, 5 seeds
# ═══════════════════════════════════════════════════════════

ablation_configs = {
    "MARS (full)":            {},
    "- Thompson Sampling":    {"override_strategy": "knowledge_based"},
    "- 6-class confidence":   {"disable_confidence": True},
    "- Knowledge Graph":      {"disable_kg": True},
    "- LSTM prediction":      {},  # use Monolithic baseline
    "- LambdaMART":           {"disable_reranking": True},
    "- DiagnosticAgent":      {"disable_irt": True},
    "- PersonalizationAgent": {"disable_clustering": True},
}

ablation_all_seeds = defaultdict(list)

for seed in SEEDS:
    print(f"\nSeed {seed}:")

    for ablation_name, config in ablation_configs.items():
        m = load_cached_metrics("ablation_v2", seed, ablation_name)

        if m is None:
            if ablation_name == "MARS (full)":
                # Reuse main results
                m = load_cached_metrics("main", seed, "MARS (ours)")
                if m is None:
                    p, s, g, ga, _, cm = run_mars(eval_pairs, train_df, seed,
                                                   pred_epochs=FAST_MARS_PRED_EPOCHS)
                    m = compute_metrics(p, g, s, ga)
                    m["confidence_f1"] = cm.get("cv_f1_macro_mean", 0.0)
                    save_cached_metrics("main", seed, "MARS (ours)", m)

            elif ablation_name == "- LSTM prediction":
                # Replace LSTM with Monolithic XGBoost (no sequence model)
                m = load_cached_metrics("main", seed, "Monolithic")
                if m is None:
                    p, s, g, ga = baseline_monolithic(eval_pairs, train_df, seed)
                    m = compute_metrics(p, g, s, ga)

            elif ablation_name == "- 6-class confidence":
                # Use Binary-conf baseline (confidence_class=0 for all)
                m = load_cached_metrics("main", seed, "Binary-conf")
                if m is None:
                    p, s, g, ga = baseline_binary_conf(eval_pairs, train_df, seed,
                                                        epochs=FAST_BINARY_CONF_EPOCHS)
                    m = compute_metrics(p, g, s, ga)

            else:
                # Real ablation: re-run MARS with component disabled
                p, s, g, ga, _, cm = run_mars(
                    eval_pairs, train_df, seed,
                    pred_epochs=FAST_MARS_PRED_EPOCHS,
                    **config,
                )
                m = compute_metrics(p, g, s, ga)
                m["confidence_f1"] = cm.get("cv_f1_macro_mean", 0.0)

            save_cached_metrics("ablation_v2", seed, ablation_name, m)

        ablation_all_seeds[ablation_name].append(m)
        print(f"  {ablation_name:30s}: NDCG@10={m['ndcg_at_10']:.4f}  AUC={m['auc_roc']:.4f}")

# --- Aggregate results ---
ablation_results = {}
metric_keys = [
    "auc_roc", "precision_at_5", "precision_at_10",
    "recall_at_5", "recall_at_10",
    "ndcg_at_5", "ndcg_at_10", "ndcg_at_20",
    "map_at_5", "map_at_10", "mrr",
    "coverage", "diversity", "novelty",
    "cold_start_acc5", "confidence_f1",
]

for name, metrics_list in ablation_all_seeds.items():
    agg = {}
    for mn in metric_keys:
        vals = [m.get(mn, 0.0) for m in metrics_list]
        agg[mn] = np.mean(vals)
        agg[f"{mn}_std"] = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
    ablation_results[name] = agg

# --- Statistical significance: paired t-test MARS(full) vs each ablation ---
full_metrics_list = ablation_all_seeds["MARS (full)"]

abl_rows = []
full_agg = ablation_results["MARS (full)"]
for name, m in ablation_results.items():
    row = {"Configuration": name}
    for mn, dn in zip(metric_keys, display_names):
        val = m[mn]
        std = m.get(f"{mn}_std", 0.0)
        delta = val - full_agg[mn]
        delta_pct = (delta / full_agg[mn] * 100) if full_agg[mn] != 0 else 0.0

        # Paired t-test
        if name != "MARS (full)" and len(ablation_all_seeds[name]) >= 2:
            full_vals = [r.get(mn, 0.0) for r in full_metrics_list]
            abl_vals = [r.get(mn, 0.0) for r in ablation_all_seeds[name]]
            diffs = np.array(full_vals) - np.array(abl_vals)
            if np.any(diffs != 0):
                _, p_val = sp_stats.ttest_rel(full_vals, abl_vals)
            else:
                p_val = 1.0
            # Cohen's d
            pooled_std = np.sqrt((np.var(full_vals, ddof=1) + np.var(abl_vals, ddof=1)) / 2)
            cohens_d = (np.mean(full_vals) - np.mean(abl_vals)) / pooled_std if pooled_std > 0 else 0.0
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        else:
            p_val, cohens_d, sig = 1.0, 0.0, ""

        if name == "MARS (full)":
            row[dn] = f"{val:.4f} +/- {std:.4f}"
        else:
            row[dn] = f"{val:.4f} +/- {std:.4f} ({delta_pct:+.1f}% {sig})"
    abl_rows.append(row)

table2 = pd.DataFrame(abl_rows)
print("\nTable 2: Ablation Study (mean +/- std over 5 seeds, real component removal)")
print("=" * 200)
print(table2[["Configuration"] + display_names].to_string(index=False))

table2.to_csv(RESULTS_DIR / "table2_ablation.csv", index=False)
print(f"\nSaved -> {RESULTS_DIR / 'table2_ablation.csv'}")

## 8. Cold-Start Evaluation

In [14]:
# Evaluate cold-start separately — across all seeds
if cold_pairs:
    print(f"Cold-start evaluation on {len(cold_pairs)} users with < 5 interactions")
    cold_all_seeds = defaultdict(list)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Random
        p, s, g, ga = baseline_random(cold_pairs, seed=seed)
        cold_all_seeds["Random"].append(compute_metrics(p, g, s, ga))

        # Popular
        p, s, g, ga = baseline_popular(cold_pairs, train_df)
        cold_all_seeds["Popular"].append(compute_metrics(p, g, s, ga))

        # MARS
        p, s, g, ga, _, _ = run_mars(cold_pairs, train_df, seed=seed)
        cold_all_seeds["MARS"].append(compute_metrics(p, g, s, ga))

    print("\nCold-Start Results (mean \u00b1 std over 5 seeds):")
    for name, metrics_list in cold_all_seeds.items():
        acc5 = [m["accuracy_at_5"] for m in metrics_list]
        ndcg5 = [m["ndcg_at_5"] for m in metrics_list]
        print(f"  {name:15s}: Acc@5={np.mean(acc5):.4f}\u00b1{np.std(acc5, ddof=1):.4f}  "
              f"NDCG@5={np.mean(ndcg5):.4f}\u00b1{np.std(ndcg5, ddof=1):.4f}")

    # Update cold_start_acc5 in main results (use mean across seeds)
    for name in cold_all_seeds:
        full_name = name if name != "MARS" else "MARS (ours)"
        if full_name in all_results:
            mean_acc5 = np.mean([m["accuracy_at_5"] for m in cold_all_seeds[name]])
            for r in all_results[full_name]:
                r["cold_start_acc5"] = round(mean_acc5, 4)
else:
    print("No cold-start users found (all have >= 5 interactions in context)")


Cold-start evaluation on 109 users with < 5 interactions


2026-04-02 21:53:37 | mars.agent.diagnostic          | INFO    | Building response matrix: 2274 questions, 429 users


mars.agent.diagnostic | Building response matrix: 2274 questions, 429 users


2026-04-02 21:53:38 | mars.agent.diagnostic          | INFO    | Calibrating IRT 3PL: 429 students x 2274 items


mars.agent.diagnostic | Calibrating IRT 3PL: 429 students x 2274 items


...

2026-04-02 22:44:12 | mars.agent.prediction          | INFO    | Epoch 5/5  train_loss=0.0781  val_loss=0.0751


mars.agent.prediction | Epoch 5/5  train_loss=0.0781  val_loss=0.0751


2026-04-02 22:44:15 | mars.agent.prediction          | INFO    | Training complete: best_epoch=4, val_loss=0.0751, val_auc=0.5538


mars.agent.prediction | Training complete: best_epoch=4, val_loss=0.0751, val_auc=0.5538



Cold-Start Results (mean ± std over 5 seeds):
  Random         : Acc@5=0.0175±0.0130  NDCG@5=0.0132±0.0076
  Popular        : Acc@5=0.1159±0.0000  NDCG@5=0.1349±0.0000
  MARS           : Acc@5=0.1159±0.0000  NDCG@5=0.1355±0.0000


## 9. Summary

In [15]:
print("\n" + "=" * 60)
print("  EVALUATION COMPLETE")
print("=" * 60)
print(f"  Methods evaluated:    {len(all_results)}")
print(f"  Seeds:                {SEEDS}")
print(f"  Test users:           {len(eval_pairs)}")
print(f"  Cold-start users:     {len(cold_pairs)}")
print(f"\n  Files saved:")
print(f"    {RESULTS_DIR / 'table1_comparison.csv'}")
print(f"    {RESULTS_DIR / 'table2_ablation.csv'}")
print(f"    {RESULTS_DIR / 'statistical_significance.csv'}")
print("=" * 60)


  EVALUATION COMPLETE
  Methods evaluated:    10
  Seeds:                [42, 123, 456, 789, 2024]
  Test users:           230
  Cold-start users:     109

  Files saved:
    results\table1_comparison.csv
    results\table2_ablation.csv
    results\statistical_significance.csv
